# Cisco CX CDC: Ibis / DuckDB (Portable)

**Source**: S3 Parquet (10M rows)  
**CDC**: Hash-based diff with Ibis (DuckDB backend)  
**Target**: Snowflake table  
**Runs on**: Snowflake SPCS and EC2/Docker identically  

Ibis compiles the same DataFrame API to DuckDB (local) or Snowflake (cloud).

In [1]:
import importlib
required = [('ibis-framework', 'ibis'), ('duckdb', 'duckdb'), ('pyarrow', 'pyarrow'), ('s3fs', 's3fs')]
missing = []
for pkg, mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)
if missing:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing + ['-q'])
    print(f"Installed: {missing}")
else:
    print("All packages available.")

All packages available.


In [2]:
import os, json

# Load AWS credentials from Snowflake credential store (no hardcoded keys)
def get_snowflake_connection():
    """Connect to Snowflake - works in SPCS or on-prem."""
    try:
        from snowflake.snowpark.context import get_active_session
        session = get_active_session()
        print("Connected via Snowflake SPCS")
        return session, "SPCS"
    except:
        pass
    import snowflake.connector
    conn = snowflake.connector.connect(connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME", "default"))
    print(f"Connected locally to {conn.account}")
    return conn, "LOCAL"

sf_conn, RUNTIME = get_snowflake_connection()
result = sf_conn.sql("SELECT creds FROM CISCO_CX_PILOT.SECRETS.AWS_CREDS WHERE id = 1").collect() if RUNTIME == 'SPCS' else None
if RUNTIME == 'LOCAL':
    cur = sf_conn.cursor()
    cur.execute("SELECT creds FROM CISCO_CX_PILOT.SECRETS.AWS_CREDS WHERE id = 1")
    creds_json = cur.fetchone()[0]
else:
    creds_json = result[0][0]

creds = json.loads(creds_json)
os.environ['AWS_ACCESS_KEY_ID'] = creds['aws_access_key_id']
os.environ['AWS_SECRET_ACCESS_KEY'] = creds['aws_secret_access_key']
os.environ['AWS_SESSION_TOKEN'] = creds.get('aws_session_token', '')

print(f'Runtime: {RUNTIME}')
print('AWS credentials loaded from Snowflake credential store.')

Connected locally to FZB62295


Runtime: LOCAL
AWS credentials loaded from Snowflake credential store.


In [3]:
import time
import pyarrow.parquet as pq
import pyarrow.fs as pafs
import pandas as pd
import numpy as np

S3_BUCKET = "cisco-cx-cdc-pilot"
TARGET_DB = 'CISCO_CX_PILOT'
TARGET_SCHEMA = 'PROCESSED'
PREV_TABLE = 'PREV_SNAPSHOT_STAGING'

COMPARE_COLS = ['software_version', 'cpu_utilization', 'memory_utilization',
                'critical_bugs_count', 'contract_status', 'ip_address']

# Build S3FileSystem with credentials
aws_key = os.environ.get('AWS_ACCESS_KEY_ID', '')
aws_secret = os.environ.get('AWS_SECRET_ACCESS_KEY', '')
aws_token = os.environ.get('AWS_SESSION_TOKEN', '')

if aws_key and aws_secret:
    fs = pafs.S3FileSystem(
        region='us-east-1',
        access_key=aws_key,
        secret_key=aws_secret,
        session_token=aws_token if aws_token else None
    )
else:
    fs = pafs.S3FileSystem(region='us-east-1')

# --- PREP: Read prev_snapshot from S3 and load into Snowflake ---
t0 = time.time()
print('Reading prev_snapshot from S3...')
prev_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/prev_snapshot", filesystem=fs).to_pandas()
prev_count = len(prev_pdf)
t_read_prev_s3 = time.time() - t0
print(f'  prev_snapshot read from S3: {prev_count:,} rows in {t_read_prev_s3:.1f}s')

# Load prev_snapshot into Snowflake
t0 = time.time()
prev_pdf_upper = prev_pdf.copy()
prev_pdf_upper.columns = [c.upper() for c in prev_pdf_upper.columns]

if RUNTIME == 'LOCAL':
    from snowflake.connector.pandas_tools import write_pandas
    sf_conn.cursor().execute(f'USE DATABASE {TARGET_DB}')
    sf_conn.cursor().execute(f'USE SCHEMA {TARGET_SCHEMA}')
    sf_conn.cursor().execute(f"""CREATE TABLE IF NOT EXISTS {PREV_TABLE} (
        DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR,
        CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER,
        CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)""")
    sf_conn.cursor().execute(f'TRUNCATE TABLE {PREV_TABLE}')
    success, nchunks, nrows, _ = write_pandas(sf_conn, prev_pdf_upper, PREV_TABLE, database=TARGET_DB, schema=TARGET_SCHEMA)
else:
    sf_conn.sql(f'USE DATABASE {TARGET_DB}').collect()
    sf_conn.sql(f'USE SCHEMA {TARGET_SCHEMA}').collect()
    sf_conn.write_pandas(prev_pdf_upper, PREV_TABLE, database=TARGET_DB, schema=TARGET_SCHEMA, overwrite=True)

t_load_sf = time.time() - t0
print(f'  prev_snapshot loaded into Snowflake ({TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}): {t_load_sf:.1f}s')
print(f'\n--- Prep complete (not included in pipeline time) ---')

Reading prev_snapshot from S3...


  prev_snapshot read from S3: 10,000,000 rows in 10.1s


  prev_snapshot loaded into Snowflake (CISCO_CX_PILOT.PROCESSED.PREV_SNAPSHOT_STAGING): 124.0s

--- Prep complete (not included in pipeline time) ---


In [4]:
import ibis
ibis.options.interactive = True
print(f'Ibis {ibis.__version__}')

# --- PIPELINE START: Read prev from Snowflake + Read curr from S3 + CDC ---
t_pipeline_start = time.time()

# Read prev_snapshot back from Snowflake
t0 = time.time()
if RUNTIME == 'LOCAL':
    cur = sf_conn.cursor()
    cur.execute(f'SELECT * FROM {TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}')
    prev_from_sf = cur.fetch_pandas_all()
else:
    prev_from_sf = sf_conn.sql(f'SELECT * FROM {TARGET_DB}.{TARGET_SCHEMA}.{PREV_TABLE}').to_pandas()

prev_from_sf.columns = [c.lower() for c in prev_from_sf.columns]
t_read_sf = time.time() - t0
print(f'  prev_snapshot read from Snowflake: {len(prev_from_sf):,} rows in {t_read_sf:.1f}s')

# Read curr_snapshot from S3
t0 = time.time()
curr_pdf = pq.read_table(f"{S3_BUCKET}/cdc_data/curr_snapshot", filesystem=fs).to_pandas()
t_read_curr_s3 = time.time() - t0
print(f'  curr_snapshot read from S3: {len(curr_pdf):,} rows in {t_read_curr_s3:.1f}s')

print(f'\n--- Data Ready for CDC ---')
print(f'  Previous (from Snowflake): {len(prev_from_sf):,}')
print(f'  Current  (from S3):        {len(curr_pdf):,}')

# --- Ibis/DuckDB CDC ---
t0 = time.time()
con = ibis.duckdb.connect()
prev_tbl = con.create_table('prev_snapshot', prev_from_sf)
curr_tbl = con.create_table('curr_snapshot', curr_pdf)
t_create = time.time() - t0
print(f'\nTables loaded into DuckDB: {t_create:.1f}s')

t0 = time.time()
compare_expr_p = ibis.literal('|').join([prev_tbl[c].cast('string') for c in COMPARE_COLS])
prev_hashed = prev_tbl.mutate(_hash=compare_expr_p.hash())

compare_expr_c = ibis.literal('|').join([curr_tbl[c].cast('string') for c in COMPARE_COLS])
curr_hashed = curr_tbl.mutate(_hash=compare_expr_c.hash())

prev_keys = prev_hashed.select('device_id', _hash_prev=prev_hashed._hash)
curr_keys = curr_hashed.select('device_id', _hash_curr=curr_hashed._hash)

merged = curr_keys.outer_join(prev_keys, 'device_id')

n_inserts = merged.filter(merged._hash_prev.isnull()).count().execute()
n_deletes = merged.filter(merged._hash_curr.isnull()).count().execute()
both = merged.filter(merged._hash_curr.notnull() & merged._hash_prev.notnull())
n_updates = both.filter(both._hash_curr != both._hash_prev).count().execute()
t_cdc = time.time() - t0

print(f'\n=== Ibis (DuckDB backend) CDC Results ===')
print(f'  DuckDB Table Load:      {t_create:.1f}s')
print(f'  Hash + Join + Classify: {t_cdc:.1f}s')
print(f'  Inserts: {n_inserts:,} | Updates: {n_updates:,} | Deletes: {n_deletes:,}')

Ibis 12.0.0


  prev_snapshot read from Snowflake: 10,000,000 rows in 17.5s


  curr_snapshot read from S3: 10,020,000 rows in 10.4s

--- Data Ready for CDC ---
  Previous (from Snowflake): 10,000,000
  Current  (from S3):        10,020,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Tables loaded into DuckDB: 23.0s



=== Ibis (DuckDB backend) CDC Results ===
  DuckDB Table Load:      23.0s
  Hash + Join + Classify: 3.2s
  Inserts: 25,000 | Updates: 18,952 | Deletes: 5,000


In [5]:
t0 = time.time()

insert_ids = merged.filter(merged._hash_prev.isnull()).select('device_id')
update_ids = both.filter(both._hash_curr != both._hash_prev).select('device_id')
all_change_ids = ibis.union(insert_ids, update_ids)

upsert_data = curr_tbl.filter(curr_tbl.device_id.isin(all_change_ids.device_id)).execute()
upsert_data.columns = [c.upper() for c in upsert_data.columns]

if RUNTIME == "LOCAL":
    from snowflake.connector.pandas_tools import write_pandas
    sf_conn.cursor().execute("USE DATABASE CISCO_CX_PILOT")
    sf_conn.cursor().execute("USE SCHEMA PROCESSED")
    sf_conn.cursor().execute("CREATE TABLE IF NOT EXISTS IBIS_CDC_RESULT (DEVICE_ID VARCHAR, CUSTOMER_ID VARCHAR, SOFTWARE_VERSION VARCHAR, CPU_UTILIZATION FLOAT, MEMORY_UTILIZATION FLOAT, CRITICAL_BUGS_COUNT INTEGER, CONTRACT_STATUS VARCHAR, IP_ADDRESS VARCHAR, PRODUCT_FAMILY VARCHAR)")
    sf_conn.cursor().execute("TRUNCATE TABLE IBIS_CDC_RESULT")
    success, nchunks, nrows, _ = write_pandas(sf_conn, upsert_data, 'IBIS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED')
    print(f'Wrote {nrows:,} rows to Snowflake in {time.time()-t0:.1f}s')
else:
    sf_conn.sql("USE DATABASE CISCO_CX_PILOT").collect()
    sf_conn.sql("USE SCHEMA PROCESSED").collect()
    sf_conn.write_pandas(upsert_data, 'IBIS_CDC_RESULT', database='CISCO_CX_PILOT', schema='PROCESSED', overwrite=True)
    print(f'Wrote {len(upsert_data):,} rows to Snowflake in {time.time()-t0:.1f}s')

t_write = time.time() - t0
t_total = time.time() - t_pipeline_start
con.disconnect()

print(f'\n=== TOTAL PIPELINE SUMMARY ===')
print(f'  Read prev from Snowflake:  {t_read_sf:.1f}s')
print(f'  Read curr from S3:         {t_read_curr_s3:.1f}s')
print(f'  DuckDB Table Load:         {t_create:.1f}s')
print(f'  Hash + Join + Classify:    {t_cdc:.1f}s')
print(f'  Write results to SF:       {t_write:.1f}s')
print(f'  TOTAL PIPELINE:            {t_total:.1f}s')

Wrote 43,952 rows to Snowflake in 6.9s

=== TOTAL PIPELINE SUMMARY ===
  Read prev from Snowflake:  17.5s
  Read curr from S3:         10.4s
  DuckDB Table Load:         23.0s
  Hash + Join + Classify:    3.2s
  Write results to SF:       6.9s
  TOTAL PIPELINE:            61.0s
